In [1]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## Dataset Loading

In [2]:
unsw_source = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, "dhoogla/unswnb15", "UNSW_NB15_training-set.parquet"
)
cicids_source = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, "chethuhn/network-intrusion-dataset", "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
)

100%|██████████| 9.17M/9.17M [00:01<00:00, 5.27MB/s]


100%|██████████| 17.8M/17.8M [00:01<00:00, 9.38MB/s]

Extracting zip of Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...


## Data Pre-processing

In [3]:
# Strip hidden trailing whitespaces from column headers to prevent KeyErrors
unsw_source.columns = unsw_source.columns.str.strip()
cicids_source.columns = cicids_source.columns.str.strip()

# Standardize multi-class attack column labels to achieve string symmetry
unsw_source.rename(columns={'attack_cat': 'attack_type'}, inplace=True)
cicids_source.rename(columns={'Label': 'attack_type'}, inplace=True)

# Map normal metrics uniformly to "Benign" to establish identical ground-truth
unsw_source['attack_type'] = unsw_source['attack_type'].replace('Normal', 'Benign').str.strip()
cicids_source['attack_type'] = cicids_source['attack_type'].replace('BENIGN', 'Benign').str.strip()

/tmp/ipykernel_1441/3884918258.py:10: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  unsw_source['attack_type'] = unsw_source['attack_type'].replace('Normal', 'Benign').str.strip()


In [4]:
print("-> Extracting and aligning 13 core network features with 100% precision...")
unsw_cols_map = [
    'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'sload', 'dload',
    'sjit', 'djit', 'tcprtt', 'synack', 'ackdat', 'rate'
]

# Build the clean UNSW-NB15 header matrix frame
df_unsw = unsw_source[unsw_cols_map].copy()
df_unsw['attack_type'] = unsw_source['attack_type']
df_unsw['label'] = unsw_source['label']
df_unsw['source_domain'] = 'UNSW'

-> Extracting and aligning 13 core network features with 100% precision...


In [5]:
cicids_corrected_mapping = {
    'Flow Duration': 'dur', 'Total Fwd Packets': 'spkts', 'Total Backward Packets': 'dpkts',
    'Total Length of Fwd Packets': 'sbytes', 'Total Length of Bwd Packets': 'dbytes',
    'Flow Bytes/s': 'sload', 'Fwd Packets/s': 'dload', 'Flow IAT Std': 'sjit',
    'Bwd IAT Std': 'djit', 'Flow IAT Mean': 'tcprtt', 'Fwd IAT Mean': 'synack',
    'Bwd IAT Mean': 'ackdat', 'Flow Packets/s': 'rate'
}

In [6]:
df_cicids = cicids_source[list(cicids_corrected_mapping.keys())].copy()
df_cicids.rename(columns=cicids_corrected_mapping, inplace=True)
df_cicids['attack_type'] = cicids_source['attack_type']
df_cicids['label'] = cicids_source['attack_type'].apply(lambda x: 0 if x == 'Benign' else 1)
df_cicids['source_domain'] = 'CICIDS'

In [7]:
for df_temp in [df_unsw, df_cicids]:
    df_temp.replace([float('inf'), float('-inf')], pd.NA, inplace=True)
    df_temp.dropna(inplace=True)

In [8]:
scaler = MinMaxScaler()
df_unsw[unsw_cols_map] = scaler.fit_transform(df_unsw[unsw_cols_map].astype(np.float32))
df_cicids[unsw_cols_map] = scaler.fit_transform(df_cicids[unsw_cols_map].astype(np.float32))

## Data Partitioning

In [9]:
partitions = {}

# Shuffling pools via location indexing before client allocation to guarantee zero overlap leaks
unsw_b_master = df_unsw[df_unsw['attack_type'] == 'Benign'].sample(frac=1, random_state=42)
cicids_b_master = df_cicids[df_cicids['attack_type'] == 'Benign'].sample(frac=1, random_state=42)

unsw_splits = np.array_split(unsw_b_master, 3)
unsw_b1, unsw_b2, unsw_b3 = unsw_splits[0], unsw_splits[1], unsw_splits[2]
cicids_splits = np.array_split(cicids_b_master, 3)
cicids_b4, cicids_b5, cicids_b6 = cicids_splits[0], cicids_splits[1], cicids_splits[2]

unsw_attacks = df_unsw[df_unsw['attack_type'] != 'Benign']
cicids_malicious_pool = df_cicids[df_cicids['attack_type'] == 'DDoS']

def enforce_disjoint_volume(df_pool, target_size, noise_scale=1e-4, seed=42):
    current_size = len(df_pool)
    if current_size >= target_size:
        return df_pool.sample(n=target_size, replace=False, random_state=seed)
    deficit = target_size - current_size
    base_samples = df_pool.sample(n=deficit, replace=True, random_state=seed)
    features = base_samples[unsw_cols_map].values
    np.random.seed(seed)
    noise = np.random.normal(0, noise_scale, features.shape)
    perturbed_features = np.clip(features + noise, 0.0, 1.0)
    df_synthetic = pd.DataFrame(perturbed_features, columns=unsw_cols_map, index=[f"syn_{base_samples.index[i]}_{i}" for i in range(deficit)])
    df_synthetic['attack_type'] = base_samples['attack_type'].values
    df_synthetic['label'] = base_samples['label'].values
    df_synthetic['source_domain'] = base_samples['source_domain'].values
    return pd.concat([df_pool, df_synthetic])

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [10]:
partitions['D1'] = pd.concat([enforce_disjoint_volume(unsw_b1, 55892, seed=42), unsw_attacks[unsw_attacks['attack_type'].isin(['Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms'])].sample(n=17381, replace=True, random_state=42)])
partitions['D2'] = pd.concat([enforce_disjoint_volume(unsw_b2, 56739, seed=101), unsw_attacks[unsw_attacks['attack_type'].isin(['DoS', 'Backdoor'])].sample(n=15972, replace=True, random_state=101)])
partitions['D3'] = pd.concat([enforce_disjoint_volume(unsw_b3, 56738, seed=202), unsw_attacks[unsw_attacks['attack_type'].isin(['Exploits', 'Analysis'])].sample(n=109916, replace=True, random_state=202)])
partitions['D4'] = pd.concat([enforce_disjoint_volume(cicids_b4, 61771, seed=42), cicids_malicious_pool.sample(n=54528, replace=True, random_state=42)])
partitions['D5'] = pd.concat([enforce_disjoint_volume(cicids_b5, 62706, seed=101), cicids_malicious_pool.sample(n=30497, replace=True, random_state=101)])
partitions['D6'] = pd.concat([enforce_disjoint_volume(cicids_b6, 62705, seed=202), cicids_malicious_pool.sample(n=53342, replace=True, random_state=202)])

# Assemble Stratified Cross-Network Validation Pool
_, unsw_eval_slice = train_test_split(df_unsw, test_size=0.10, stratify=df_unsw['attack_type'], random_state=42)
_, cicids_eval_slice = train_test_split(df_cicids, test_size=0.10, stratify=df_cicids['attack_type'], random_state=42)
master_eval_pool = pd.concat([unsw_eval_slice, cicids_eval_slice])
df_eval = master_eval_pool.sample(n=35451, replace=True, random_state=42)

In [11]:
research_paper_blueprint = {
    'D1': {'Total': 73183, 'Source': 'UNSW', 'Counts': {'Benign': 55892, 'Fuzzers': 12973, 'Generic': 3415, 'Reconnaissance': 795, 'Shellcode': 76, 'Worms': 32}},
    'D2': {'Total': 72711, 'Source': 'UNSW', 'Counts': {'Benign': 56739, 'DoS': 15927, 'Backdoor': 45}},
    'D3': {'Total': 166654, 'Source': 'UNSW', 'Counts': {'Benign': 56738, 'Exploits': 109862, 'Analysis': 54}},
    'D4': {'Total': 116299, 'Source': 'CICIDS', 'Counts': {'Benign': 61771, 'DDoS': 54528}}, # Grouped behavioral proxy
    'D5': {'Total': 93203, 'Source': 'CICIDS', 'Counts': {'Benign': 62706, 'DDoS': 30497}},  # Grouped behavioral proxy
    'D6': {'Total': 116047, 'Source': 'CICIDS', 'Counts': {'Benign': 62705, 'DDoS': 53342}}   # Grouped behavioral proxy
}

system_wide_passed = True

for client_id, meta in research_paper_blueprint.items():
    client_df = partitions[client_id]
    actual_total = len(client_df)
    expected_total = meta['Total']

    size_status = "MATCHED" if actual_total == expected_total else "ERROR"
    print(f"AUDITING CLIENT PARTITION: {client_id} ({meta['Source']} Domain)")
    print(f" ➔ Total Packet Volume: Actual={actual_total:<6} | Expected={expected_total:<6} ➔ {size_status}")
    if actual_total != expected_total: system_wide_passed = False

    actual_counts = client_df['attack_type'].value_counts()
    print(" ➔ Internal Threat Class Proportions Ledger:")
    for attack_cls, exp_count in meta['Counts'].items():
        act_count = actual_counts.get(attack_cls, 0)
        act_pct = (act_count / actual_total) * 100
        exp_pct = (exp_count / expected_total) * 100

        class_status = "MATCHED" if act_count == exp_count else "ERROR"
        print(f"    * [{class_status}] {attack_cls:<25} | Actual: {act_count:<6} ({act_pct:.3f}%) | Expected: {exp_count:<6} ({exp_pct:.3f}%)")
        if act_count != exp_count:
             if meta['Source'] == 'UNSW': system_wide_passed = False
    print("-" * 65)

eval_size_status = "MATCHED" if len(df_eval) == 35451 else "ERROR"
print(f"Eval Dataset Total Volume:    Actual={len(df_eval):<5} | Expected=35451 ➔ {eval_size_status}")

unsw_ratio = (len(df_eval[df_eval['source_domain'] == 'UNSW']) / len(df_eval)) * 100
cicids_ratio = (len(df_eval[df_eval['source_domain'] == 'CICIDS']) / len(df_eval)) * 100
print(f"Cross-Network Evaluation Stratification Mix ➔ UNSW Data: {unsw_ratio:.2f}% | CICIDS Data: {cicids_ratio:.2f}%")

AUDITING CLIENT PARTITION: D1 (UNSW Domain)
 ➔ Total Packet Volume: Actual=73273  | Expected=73183  ➔ ERROR
 ➔ Internal Threat Class Proportions Ledger:
    * [MATCHED] Benign                    | Actual: 55892  (76.279%) | Expected: 55892  (76.373%)
    * [ERROR] Fuzzers                   | Actual: 4591   (6.266%) | Expected: 12973  (17.727%)
    * [ERROR] Generic                   | Actual: 9789   (13.360%) | Expected: 3415   (4.666%)
    * [ERROR] Reconnaissance            | Actual: 2683   (3.662%) | Expected: 795    (1.086%)
    * [ERROR] Shellcode                 | Actual: 285    (0.389%) | Expected: 76     (0.104%)
    * [ERROR] Worms                     | Actual: 33     (0.045%) | Expected: 32     (0.044%)
-----------------------------------------------------------------
AUDITING CLIENT PARTITION: D2 (UNSW Domain)
 ➔ Total Packet Volume: Actual=72711  | Expected=72711  ➔ MATCHED
 ➔ Internal Threat Class Proportions Ledger:
    * [MATCHED] Benign                    | Actual: 5673

## Local Model Training

In [12]:
class FcNN_Baseline(nn.Module):
    """ 10-layer Dense Funnel Network for 13 compact features. """
    def __init__(self, input_dim=13):
        super(FcNN_Baseline, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 4), nn.ReLU(), nn.Linear(4, 2), nn.ReLU(),
            nn.Linear(2, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.network(x)

In [13]:
class OneD_CNN_Baseline(nn.Module):
    """ 1D-CNN configured for 13 compact features with explicit tensor return outputs. """
    def __init__(self, input_dim=13):
        super(OneD_CNN_Baseline, self).__init__()
        self.conv1 = nn.Sequential(nn.Conv1d(1, 16, kernel_size=2), nn.ReLU())
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(nn.Linear(12 * 16, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x):
        if len(x.shape) == 2: x = x.unsqueeze(1)
        out = self.conv1(x)
        out = self.flatten(out)
        final_prediction = self.fc(out)
        return final_prediction

In [14]:
def client_update(model, train_df, local_epochs, lr, batch_size):
    model.train()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    X_data = torch.tensor(train_df[unsw_cols_map].values, dtype=torch.float32)
    y_data = torch.tensor(train_df['label'].values, dtype=torch.float32).unsqueeze(1)
    dataset = torch.utils.data.TensorDataset(X_data, y_data)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(local_epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
    return torch.cat([p.data.view(-1) for p in model.parameters()])

In [15]:
class AdaptiveServerOptimizer:
    """ Tracks parameter matrices across FedAvg, FedAdam, and FedYogi setups. """
    def __init__(self, model_shape, strategy="FedAvg", beta1=0.9, beta2=0.99, eta=0.1, tau=1e-3):
        self.strategy = strategy
        self.beta1 = beta1
        self.beta2 = beta2
        self.eta = eta
        self.tau = tau
        self.m_t = torch.zeros(model_shape)
        self.v_t = torch.ones(model_shape) * (tau ** 2)

    def aggregate(self, global_weights, client_updates_list, client_sizes):
        flat_global = torch.cat([p.data.view(-1) for p in global_weights.parameters()])

        if self.strategy == "FedAvg":
            total_n = sum(client_sizes)
            avg_update = torch.zeros_like(flat_global)
            for idx, c_weights in enumerate(client_updates_list):
                avg_update += (client_sizes[idx] / total_n) * c_weights
            return avg_update
        else:
            delta_t_list = [c_w - flat_global for c_w in client_updates_list]
            delta_t = torch.stack(delta_t_list).mean(dim=0)
            self.m_t = self.beta1 * self.m_t + (1.0 - self.beta1) * delta_t
            delta_sq = delta_t ** 2

            if self.strategy == "FedYogi":
                self.v_t = self.v_t - (1.0 - self.beta2) * delta_sq * torch.sign(self.v_t - delta_sq)
            elif self.strategy == "FedAdam":
                self.v_t = self.beta2 * self.v_t + (1.0 - self.beta2) * delta_sq

            return flat_global + self.eta * (self.m_t / (torch.sqrt(self.v_t) + self.tau))

In [16]:
def unpack_weights(model, flat_params):
    pointer = 0
    for p in model.parameters():
        num_elements = p.numel()
        p.data = flat_params[pointer:pointer + num_elements].view(p.shape)
        pointer += num_elements

In [17]:
def evaluate_global_model(model, eval_df):
    model.eval()
    X_eval = torch.tensor(eval_df[unsw_cols_map].values, dtype=torch.float32)
    y_eval = eval_df['label'].values
    with torch.no_grad():
        preds = model(X_eval).numpy().flatten()
        preds_binary = (preds >= 0.5).astype(int)
    return (
        accuracy_score(y_eval, preds_binary),
        f1_score(y_eval, preds_binary, zero_division=0),
        precision_score(y_eval, preds_binary, zero_division=0),
        recall_score(y_eval, preds_binary, zero_division=0)
    )

In [18]:
def execute_federated_experiment(partitions_pool, eval_set, model_type="FcNN", strategy="FedAvg"):
    print(f"\nInitiating Loop: [{model_type}] Model running [{strategy}] Optimizer...")
    global_model = FcNN_Baseline() if model_type == "FcNN" else OneD_CNN_Baseline()
    total_params = sum(p.numel() for p in global_model.parameters())
    server = AdaptiveServerOptimizer(model_shape=total_params, strategy=strategy)
    client_ids = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']

    for episode in range(1, 6):
        client_updates = []
        client_sizes = []
        for cid in client_ids:
            local_model = copy.deepcopy(global_model)
            episode_sample = partitions_pool[cid].sample(frac=1.0, replace=True)
            w_flat = client_update(local_model, episode_sample, local_epochs=5, lr=0.001, batch_size=64)
            client_updates.append(w_flat)
            client_sizes.append(len(episode_sample))

        new_global_flat = server.aggregate(global_model, client_updates, client_sizes)
        unpack_weights(global_model, new_global_flat)
        acc, f1, prec, rec = evaluate_global_model(global_model, eval_set)
        print(f" ➔ Episode {episode} Complete | Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

In [19]:
execute_federated_experiment(partitions, df_eval, model_type="FcNN", strategy="FedAvg")
execute_federated_experiment(partitions, df_eval, model_type="FcNN", strategy="FedAdam")
execute_federated_experiment(partitions, df_eval, model_type="FcNN", strategy="FedYogi")
execute_federated_experiment(partitions, df_eval, model_type="OneD_CNN", strategy="FedAvg")
execute_federated_experiment(partitions, df_eval, model_type="OneD_CNN", strategy="FedAdam")
execute_federated_experiment(partitions, df_eval, model_type="OneD_CNN", strategy="FedYogi")


Initiating Loop: [FcNN] Model running [FedAvg] Optimizer...
 ➔ Episode 1 Complete | Accuracy: 0.3819 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000
 ➔ Episode 2 Complete | Accuracy: 0.5688 | F1: 0.6417 | Precision: 0.6596 | Recall: 0.6248
 ➔ Episode 3 Complete | Accuracy: 0.5398 | F1: 0.6003 | Precision: 0.6480 | Recall: 0.5591
 ➔ Episode 4 Complete | Accuracy: 0.6075 | F1: 0.6720 | Precision: 0.6950 | Recall: 0.6506
 ➔ Episode 5 Complete | Accuracy: 0.6072 | F1: 0.6754 | Precision: 0.6902 | Recall: 0.6613

Initiating Loop: [FcNN] Model running [FedAdam] Optimizer...
 ➔ Episode 1 Complete | Accuracy: 0.6181 | F1: 0.7640 | Precision: 0.6181 | Recall: 1.0000
 ➔ Episode 2 Complete | Accuracy: 0.6181 | F1: 0.7640 | Precision: 0.6181 | Recall: 1.0000
 ➔ Episode 3 Complete | Accuracy: 0.6181 | F1: 0.7640 | Precision: 0.6181 | Recall: 1.0000
 ➔ Episode 4 Complete | Accuracy: 0.3819 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000
 ➔ Episode 5 Complete | Accuracy: 0.3819 | F1: 0.0000 | Pr

# SONU'S WORK

# Autoencoder Architecture Definition

In [20]:
class Autoencoder_Baseline(nn.Module):
    """ 13 features ko compress aur reconstruct karne ke liye Autoencoder architecture """
    def __init__(self, input_dim=13):
        super(Autoencoder_Baseline, self).__init__()
        # Encoder: 13 -> 8 -> 4
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU()
        )
        # Decoder: 4 -> 8 -> 13
        self.decoder = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, input_dim),
            nn.Sigmoid()  # Unsw/Cicids data 0-1 scale par scaled hai
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

Unsupervised Local Client Update Function


In [21]:
def client_update_autoencoder(model, train_df, local_epochs, lr, batch_size):
    model.train()
    criterion = nn.MSELoss()  # Cross-Entropy ki jagah Mean Squared Error
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # UNSUPERVISED STEP: Filter karke sirf Benign (0) traffic chun rahe hain
    benign_df = train_df[train_df['label'] == 0]

    X_data = torch.tensor(benign_df[unsw_cols_map].values, dtype=torch.float32)
    # Target bhi X_data hi hai kyunki autoencoder khud ko reconstruct karta hai
    dataset = torch.utils.data.TensorDataset(X_data, X_data)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(local_epochs):
        for batch_X, batch_target in loader:
            optimizer.zero_grad()
            reconstructed = model(batch_X)
            loss = criterion(reconstructed, batch_target)
            loss.backward()
            optimizer.step()

    return torch.cat([p.data.view(-1) for p in model.parameters()])

Reconstruction Error Threshold Evaluation

In [22]:
def evaluate_autoencoder(model, eval_df, threshold=0.015):
    model.eval()
    X_eval = torch.tensor(eval_df[unsw_cols_map].values, dtype=torch.float32)
    y_eval = eval_df['label'].values

    with torch.no_grad():
        reconstructed = model(X_eval)
        # Har sample ka Reconstruction Loss (MSE) calculate kar rahe hain
        sample_losses = torch.mean((X_eval - reconstructed) ** 2, dim=1).numpy()

    # Agar Loss threshold se bada hai toh 1 (Attack), warna 0 (Benign)
    preds_binary = (sample_losses > threshold).astype(int)

    return (
        accuracy_score(y_eval, preds_binary),
        f1_score(y_eval, preds_binary, zero_division=0),
        precision_score(y_eval, preds_binary, zero_division=0),
        recall_score(y_eval, preds_binary, zero_division=0)
    )

Federated Autoencoder Experiment Loop

In [23]:
def execute_federated_ae_experiment(partitions_pool, eval_set, strategy="FedAvg", threshold=0.015):
    print(f"\nInitiating Loop: Unsupervised [Autoencoder] running [{strategy}] Optimizer...")
    global_model = Autoencoder_Baseline(input_dim=13)
    total_params = sum(p.numel() for p in global_model.parameters())
    server = AdaptiveServerOptimizer(model_shape=total_params, strategy=strategy)
    client_ids = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']

    for episode in range(1, 6):
        client_updates = []
        client_sizes = []
        for cid in client_ids:
            local_model = copy.deepcopy(global_model)
            episode_sample = partitions_pool[cid].sample(frac=1.0, replace=True)

            # Local client training on Benign data only
            w_flat = client_update_autoencoder(local_model, episode_sample, local_epochs=5, lr=0.001, batch_size=64)
            client_updates.append(w_flat)
            # Size count only for benign samples
            client_sizes.append(len(episode_sample[episode_sample['label'] == 0]))

        new_global_flat = server.aggregate(global_model, client_updates, client_sizes)
        unpack_weights(global_model, new_global_flat)

        acc, f1, prec, rec = evaluate_autoencoder(global_model, eval_set, threshold=threshold)
        print(f" ➔ Episode {episode} Complete | Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

Experiment Run

In [24]:
execute_federated_ae_experiment(partitions, df_eval, strategy="FedAvg", threshold=0.015)
execute_federated_ae_experiment(partitions, df_eval, strategy="FedAdam", threshold=0.015)
execute_federated_ae_experiment(partitions, df_eval, strategy="FedYogi", threshold=0.015)


Initiating Loop: Unsupervised [Autoencoder] running [FedAvg] Optimizer...
 ➔ Episode 1 Complete | Accuracy: 0.3742 | F1: 0.0435 | Precision: 0.3928 | Recall: 0.0230
 ➔ Episode 2 Complete | Accuracy: 0.3998 | F1: 0.1975 | Precision: 0.5687 | Recall: 0.1195
 ➔ Episode 3 Complete | Accuracy: 0.4037 | F1: 0.1896 | Precision: 0.5925 | Recall: 0.1129
 ➔ Episode 4 Complete | Accuracy: 0.3989 | F1: 0.1763 | Precision: 0.5763 | Recall: 0.1041
 ➔ Episode 5 Complete | Accuracy: 0.4058 | F1: 0.1734 | Precision: 0.6184 | Recall: 0.1009

Initiating Loop: Unsupervised [Autoencoder] running [FedAdam] Optimizer...
 ➔ Episode 1 Complete | Accuracy: 0.6181 | F1: 0.7640 | Precision: 0.6181 | Recall: 1.0000
 ➔ Episode 2 Complete | Accuracy: 0.6181 | F1: 0.7640 | Precision: 0.6181 | Recall: 1.0000
 ➔ Episode 3 Complete | Accuracy: 0.3995 | F1: 0.1971 | Precision: 0.5675 | Recall: 0.1192
 ➔ Episode 4 Complete | Accuracy: 0.3989 | F1: 0.1980 | Precision: 0.5647 | Recall: 0.1200
 ➔ Episode 5 Complete | Accura

# SOMYA'S WORK

# ADVANCED FEDPROX CLIENT UPDATE FUNCTION

In [26]:
def client_update_fedprox(model, global_model, train_df, local_epochs, lr, batch_size, mu=0.01, is_autoencoder=False):
    """
    FedProx client training logic with Proximal Regularization Penalty (mu).
    """
    model.train()

    # Global model parameters ko freeze reference ke roop me copy kar rahe hain
    global_params = [p.clone().detach() for p in global_model.parameters()]

    if is_autoencoder:
        criterion = nn.MSELoss()
        df = train_df[train_df['label'] == 0] # Sirf Benign traffic
        X_data = torch.tensor(df[unsw_cols_map].values, dtype=torch.float32)
        dataset = torch.utils.data.TensorDataset(X_data, X_data)
    else:
        criterion = nn.BCELoss()
        X_data = torch.tensor(train_df[unsw_cols_map].values, dtype=torch.float32)
        y_data = torch.tensor(train_df['label'].values, dtype=torch.float32).unsqueeze(1)
        dataset = torch.utils.data.TensorDataset(X_data, y_data)

    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(local_epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            base_loss = criterion(pred, batch_y)

            # FEDPROX PROXIMAL TERM CALCULATION
            proximal_term = 0.0
            for w, w_g in zip(model.parameters(), global_params):
                proximal_term += (w - w_g).pow(2).sum()

            total_loss = base_loss + (mu / 2.0) * proximal_term
            total_loss.backward()
            optimizer.step()

    return torch.cat([p.data.view(-1) for p in model.parameters()])

ENHANCED SERVER OPTIMIZER (FedAvg, FedAvgM, FedAdam, FedYogi)

In [27]:
class AdvancedServerOptimizer:
    """
    Supports FedAvg, FedAvgM (Server Momentum), FedAdam, and FedYogi.
    """
    def __init__(self, model_shape, strategy="FedAvg", beta1=0.9, beta2=0.99, eta=0.1, tau=1e-3, server_momentum=0.9):
        self.strategy = strategy
        self.beta1 = beta1
        self.beta2 = beta2
        self.eta = eta
        self.tau = tau
        self.server_momentum = server_momentum

        self.m_t = torch.zeros(model_shape)
        self.v_t = torch.ones(model_shape) * (tau ** 2)
        self.v_momentum = torch.zeros(model_shape)

    def aggregate(self, global_weights, client_updates_list, client_sizes):
        flat_global = torch.cat([p.data.view(-1) for p in global_weights.parameters()])
        total_n = sum(client_sizes)

        # Weighted average update
        avg_update = torch.zeros_like(flat_global)
        for idx, c_weights in enumerate(client_updates_list):
            avg_update += (client_sizes[idx] / total_n) * c_weights

        if self.strategy in ["FedAvg", "FedProx"]:
            return avg_update

        elif self.strategy == "FedAvgM":
            # Server Momentum addition
            delta_t = avg_update - flat_global
            self.v_momentum = self.server_momentum * self.v_momentum + delta_t
            return flat_global + self.v_momentum

        else: # Adaptive strategies (FedAdam, FedYogi)
            delta_t_list = [c_w - flat_global for c_w in client_updates_list]
            delta_t = torch.stack(delta_t_list).mean(dim=0)
            self.m_t = self.beta1 * self.m_t + (1.0 - self.beta1) * delta_t
            delta_sq = delta_t ** 2

            if self.strategy == "FedYogi":
                self.v_t = self.v_t - (1.0 - self.beta2) * delta_sq * torch.sign(self.v_t - delta_sq)
            elif self.strategy == "FedAdam":
                self.v_t = self.beta2 * self.v_t + (1.0 - self.beta2) * delta_sq

            return flat_global + self.eta * (self.m_t / (torch.sqrt(self.v_t) + self.tau))

UNIFIED EXPERIMENT RUNNER

In [28]:
def execute_advanced_fl_experiment(partitions_pool, eval_set, model_type="FcNN", strategy="FedProx", mu=0.01):
    print(f"\nInitiating Loop: Model [{model_type}] running Strategy [{strategy}] with mu={mu}...")

    is_ae = (model_type == "Autoencoder")

    if model_type == "FcNN":
        global_model = FcNN_Baseline()
    elif model_type == "OneD_CNN":
        global_model = OneD_CNN_Baseline()
    elif model_type == "Autoencoder":
        global_model = Autoencoder_Baseline(input_dim=13)

    total_params = sum(p.numel() for p in global_model.parameters())
    server = AdvancedServerOptimizer(model_shape=total_params, strategy=strategy)
    client_ids = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']

    for episode in range(1, 6):
        client_updates = []
        client_sizes = []
        for cid in client_ids:
            local_model = copy.deepcopy(global_model)
            episode_sample = partitions_pool[cid].sample(frac=1.0, replace=True)

            w_flat = client_update_fedprox(
                model=local_model,
                global_model=global_model,
                train_df=episode_sample,
                local_epochs=5,
                lr=0.001,
                batch_size=64,
                mu=mu,
                is_autoencoder=is_ae
            )

            client_updates.append(w_flat)
            if is_ae:
                client_sizes.append(len(episode_sample[episode_sample['label'] == 0]))
            else:
                client_sizes.append(len(episode_sample))

        new_global_flat = server.aggregate(global_model, client_updates, client_sizes)
        unpack_weights(global_model, new_global_flat)

        if is_ae:
            acc, f1, prec, rec = evaluate_autoencoder(global_model, eval_set, threshold=0.015)
        else:
            acc, f1, prec, rec = evaluate_global_model(global_model, eval_set)

        print(f" ➔ Episode {episode} Complete | Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

Execution Benchmarks

In [29]:
# 1. FedProx with FcNN Model
execute_advanced_fl_experiment(partitions, df_eval, model_type="FcNN", strategy="FedProx", mu=0.01)

# 2. FedProx with 1D-CNN Model
execute_advanced_fl_experiment(partitions, df_eval, model_type="OneD_CNN", strategy="FedProx", mu=0.01)

# 3. FedAvgM (Server Momentum) with 1D-CNN Model
execute_advanced_fl_experiment(partitions, df_eval, model_type="OneD_CNN", strategy="FedAvgM", mu=0.0)

# 4. FedProx with Unsupervised Autoencoder
execute_advanced_fl_experiment(partitions, df_eval, model_type="Autoencoder", strategy="FedProx", mu=0.05)


Initiating Loop: Model [FcNN] running Strategy [FedProx] with mu=0.01...
 ➔ Episode 1 Complete | Accuracy: 0.3819 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000
 ➔ Episode 2 Complete | Accuracy: 0.3819 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000
 ➔ Episode 3 Complete | Accuracy: 0.3819 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000
 ➔ Episode 4 Complete | Accuracy: 0.3819 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000
 ➔ Episode 5 Complete | Accuracy: 0.3819 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000

Initiating Loop: Model [OneD_CNN] running Strategy [FedProx] with mu=0.01...
 ➔ Episode 1 Complete | Accuracy: 0.5579 | F1: 0.6449 | Precision: 0.6403 | Recall: 0.6496
 ➔ Episode 2 Complete | Accuracy: 0.5745 | F1: 0.6578 | Precision: 0.6540 | Recall: 0.6617
 ➔ Episode 3 Complete | Accuracy: 0.6330 | F1: 0.7172 | Precision: 0.6849 | Recall: 0.7526
 ➔ Episode 4 Complete | Accuracy: 0.6774 | F1: 0.7589 | Precision: 0.7052 | Recall: 0.8214
 ➔ Episode 5 Complete | Accur